# 7R-B. Build the Environment B feature representations

This notebook converts one 7R-A output package into the technical and organisational representations used by the Environment B evaluation. The supplied saved run reads `B1_structured`, final seed 2105.

It checks the source-file hashes before construction. The generator's private latent user-day plan is not read. The malicious answer key is loaded only after the feature representations have been built.

| Configuration | Representation |
|---|---|
| M0-B | 41 technical features: 29 same-day features and 12 self-history deviations. |
| M1-B | M0-B plus training-fitted raw organisational encodings. |
| M2-B | M0-B plus 18 curated context features. |
| M3-B | M0-B plus the corresponding context features after monthly role–department shuffling. |

The curated block contains one access proxy, two category-frequency features, three hours features and twelve peer deviations. Learned encodings and profiles use training data.

The generator emits `local` and `to_removable` file directions. `file_from_removable` remains in the common schema but is structurally zero in this supplied run.

**Outputs:** feature tables, separate labels, fitted profiles, the feature contract, summary, checks and hashes. No detector is trained and no H1-B decision is made here.

*Documentation copy, 8 November 2026. Notes edited for clarity; code, saved outputs and execution metadata are unchanged. This copy has not been rerun.*


In [1]:

# ============================================================
# 0. Imports + discover 7R-A input
# ============================================================
import os, glob, json, hashlib, shutil, zipfile, gc, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

WORK = Path('/kaggle/working')
OUT = WORK / 'retail_7rb'
QA = OUT / 'qa'
for p in (OUT, QA): p.mkdir(parents=True, exist_ok=True)

START_DATE = pd.Timestamp('2026-01-01')
TRAIN_END_DAY = 90
VAL_END_DAY = 120
M3_SEED = 42
AH_START, AH_END = 7, 18

# Prefer an extracted 7R-A output folder.
manifests = sorted(glob.glob('/kaggle/input/**/generator_run_manifest.json', recursive=True))
SRC = None
for m in manifests:
    candidate = Path(m).parent
    if (candidate/'raw'/'logon.parquet').exists() and (candidate/'output_sha256.csv').exists():
        SRC = candidate; break

# If Kaggle exposes only the generated ZIP, extract it to working.
if SRC is None:
    zips = sorted(glob.glob('/kaggle/input/**/retail_env_b_v1_*.zip', recursive=True))
    assert zips, 'Attach the output ZIP/folder created by Notebook 7R-A.'
    EX = WORK/'7ra_input_extracted'
    if EX.exists(): shutil.rmtree(EX)
    EX.mkdir(parents=True)
    with zipfile.ZipFile(zips[0]) as z: z.extractall(EX)
    if (EX/'generator_run_manifest.json').exists():
        SRC = EX
    else:
        hits=list(EX.rglob('generator_run_manifest.json'))
        assert hits, 'Could not locate generator_run_manifest.json after extraction.'
        SRC=hits[0].parent

print('7R-A source:', SRC)
manifest=json.load(open(SRC/'generator_run_manifest.json'))
print(json.dumps(manifest, indent=2))
assert manifest.get('model_trained') is False
assert manifest.get('h1_b_verdict') is None

R62_M0_FEATURES = [
    'dow','is_weekend',
    'logon_count','logoff_count','logon_pcs','logon_afterhours',
    'first_logon_hour','last_logoff_hour',
    'device_connects','device_pcs','device_afterhours',
    'file_events','file_distinct','file_to_removable','file_from_removable',
    'file_afterhours','file_archives','file_docs','file_exec','file_images',
    'emails_sent','email_recipients','email_external','email_attachments',
    'email_size','email_afterhours',
    'http_count','http_afterhours','http_distinct_domains',
    'logon_count_z','logon_afterhours_z','logon_pcs_z',
    'device_connects_z','file_events_z','file_to_removable_z',
    'emails_sent_z','email_external_z','email_size_z',
    'http_count_z','http_afterhours_z','http_distinct_domains_z'
]
SAME_DAY_FEATURES = R62_M0_FEATURES[:29]
BASE_FEATS = [
    'logon_count','logon_afterhours','logon_pcs','device_connects',
    'file_events','file_to_removable','emails_sent','email_external',
    'email_size','http_count','http_afterhours','http_distinct_domains'
]
assert len(R62_M0_FEATURES)==41 and len(SAME_DAY_FEATURES)==29 and len(BASE_FEATS)==12


7R-A source: /kaggle/input/notebooks/thandondhlovu10/7r-a-controlled-retail-and-distribution-environm/retail_env_b_v1_1_final_B1_structured_seed2105
{
  "design_version": "1.1",
  "scenario_mix_policy": "exact_60_20_20_by_account",
  "null_condition_role_signal_alpha": 1.0,
  "run_mode": "FINAL",
  "condition": "B1_structured",
  "seed": 2105,
  "employees": 2000,
  "active_user_days": 264893,
  "validation_malicious_accounts": 30,
  "test_malicious_accounts": 60,
  "malicious_user_days": 270,
  "model_trained": false,
  "h1_b_verdict": null
}


## 1. Verify the source package

Compare the 7R-A files with their recorded SHA-256 hashes before reading the feature data. A matching hash identifies the supplied file contents; it does not establish the timing of a protocol decision.

The check reports the private latent plan's presence but does not use its contents to construct features.


In [2]:

# ============================================================
# 1. Verify every recorded 7R-A SHA-256 hash
# ============================================================
def sha256_file(path, block=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as fh:
        while True:
            b=fh.read(block)
            if not b: break
            h.update(b)
    return h.hexdigest()

src_hashes=pd.read_csv(SRC/'output_sha256.csv')
verify=[]
for r in src_hashes.itertuples(index=False):
    p=SRC/str(r.relative_path)
    ok=p.exists() and p.stat().st_size==int(r.size_bytes) and sha256_file(p)==str(r.sha256)
    verify.append({'relative_path':r.relative_path,'passed':bool(ok)})
ver=pd.DataFrame(verify)
ver.to_csv(QA/'7ra_input_hash_verification.csv',index=False)
print(f"7R-A hashes: {int(ver.passed.sum())}/{len(ver)} passed")
assert ver.passed.all(), '7R-A source hash verification failed.'

# Explicitly prove the private latent plan exists only as archived QA; it is not loaded below.
print('Private latent plan present:', (SRC/'qa'/'latent_user_day_plan_PRIVATE.parquet').exists())
print('This notebook does not read that file.')


7R-A hashes: 25/25 passed
Private latent plan present: True
This notebook does not read that file.


## 2. Aggregate the technical channels

Read the raw logon, device, file, email and HTTP records and summarise them by active user-day. HTTP is processed one monthly file at a time.

Only the activity channels are used at this stage. The answer key and scenario identities are not loaded for these calculations.


In [3]:

# ============================================================
# 2a. Helpers
# ============================================================
def prep_time(df):
    df=df.copy()
    df['date']=pd.to_datetime(df['date'])
    df['day']=df['date'].dt.normalize()
    df['hour']=df['date'].dt.hour.astype('int16')
    df['afterhours']=((df['hour']<AH_START)|(df['hour']>=AH_END)).astype('int8')
    df['user']=df['user'].astype(str)
    return df

def recipient_count(s):
    s=s.fillna('').astype(str).str.strip()
    return np.where(s.eq(''),0,s.str.count(';')+1).astype('int32')

def external_recipient_count(s, internal_domain='retail-env-b.example'):
    s=s.fillna('').astype(str)
    out=np.zeros(len(s),dtype='int32')
    for i,x in enumerate(s):
        rec=[r.strip().lower() for r in x.split(';') if r.strip()]
        out[i]=sum(not r.endswith('@'+internal_domain) for r in rec)
    return out

print('helpers ready')


helpers ready


In [4]:

# ============================================================
# 2b. Logon, device, file
# ============================================================
logon=prep_time(pd.read_parquet(SRC/'raw'/'logon.parquet'))
logon['act']=logon['activity'].astype(str).str.lower()
g=logon.groupby(['user','day'],observed=True)
logon_feats=pd.DataFrame({
    'logon_count':g['act'].apply(lambda s:int((s=='logon').sum())),
    'logoff_count':g['act'].apply(lambda s:int((s=='logoff').sum())),
    'logon_pcs':g['pc'].nunique(),
    'logon_afterhours':g.apply(lambda d:int(d.loc[d['act']=='logon','afterhours'].sum())),
    'first_logon_hour':g.apply(lambda d:d.loc[d['act']=='logon','hour'].min()),
    'last_logoff_hour':g.apply(lambda d:d.loc[d['act']=='logoff','hour'].max()),
}).reset_index()
del logon,g; gc.collect()

device=prep_time(pd.read_parquet(SRC/'raw'/'device.parquet'))
device['act']=device['activity'].astype(str).str.lower()
g=device.groupby(['user','day'],observed=True)
device_feats=pd.DataFrame({
    'device_connects':g['act'].apply(lambda s:int((s=='connect').sum())),
    'device_pcs':g['pc'].nunique(),
    'device_afterhours':g.apply(lambda d:int(d.loc[d['act']=='connect','afterhours'].sum())),
}).reset_index()
del device,g; gc.collect()

f=prep_time(pd.read_parquet(SRC/'raw'/'file.parquet'))
ext=f['filename'].astype(str).str.lower().str.extract(r'(\.[a-z0-9]+)$',expand=False)
f['ext_group']=np.select([
    ext.isin(['.doc','.docx','.pdf','.txt','.xlsx','.csv']),
    ext.isin(['.zip','.rar','.7z']),
    ext.isin(['.exe','.dll','.bat','.sh']),
    ext.isin(['.jpg','.jpeg','.png','.bmp','.gif'])],
    ['docs','archives','exec','images'],default='other')
f['to_removable']=(f['direction'].astype(str).str.lower()=='to_removable').astype('int8')
f['from_removable']=(f['direction'].astype(str).str.lower()=='from_removable').astype('int8')
g=f.groupby(['user','day'],observed=True)
file_feats=pd.DataFrame({
    'file_events':g.size(),
    'file_distinct':g['filename'].nunique(),
    'file_to_removable':g['to_removable'].sum(),
    'file_from_removable':g['from_removable'].sum(),
    'file_afterhours':g['afterhours'].sum(),
}).reset_index()
extc=(f.pivot_table(index=['user','day'],columns='ext_group',values='filename',aggfunc='size',fill_value=0).reset_index())
for c in ['archives','docs','exec','images']:
    if c not in extc.columns: extc[c]=0
extc=extc.rename(columns={'archives':'file_archives','docs':'file_docs','exec':'file_exec','images':'file_images'})
file_feats=file_feats.merge(extc[['user','day','file_archives','file_docs','file_exec','file_images']],on=['user','day'],how='left')
del f,ext,extc,g; gc.collect()

print('logon days:',len(logon_feats),'device days:',len(device_feats),'file days:',len(file_feats))


logon days: 264893 device days: 72070 file days: 246279


In [5]:

# ============================================================
# 2c. Email
# ============================================================
e=prep_time(pd.read_parquet(SRC/'raw'/'email.parquet'))
e['recipients']=0
e['external']=0
for c in ['to','cc','bcc']:
    if c in e.columns:
        e['recipients'] += recipient_count(e[c])
        e['external'] += external_recipient_count(e[c])
e['attachments_n']=pd.to_numeric(e.get('attachments',0),errors='coerce').fillna(0).astype('int32')
e['size_n']=pd.to_numeric(e.get('size',0),errors='coerce').fillna(0)
email_feats=(e.groupby(['user','day'],observed=True)
             .agg(emails_sent=('day','size'),email_recipients=('recipients','sum'),
                  email_external=('external','sum'),email_attachments=('attachments_n','sum'),
                  email_size=('size_n','sum'),email_afterhours=('afterhours','sum')).reset_index())
del e; gc.collect()
print('email days:',len(email_feats))


email days: 258473


In [6]:

# ============================================================
# 2d. HTTP — aggregate one month at a time
# ============================================================
count_parts=[]; domain_parts=[]
http_files=sorted((SRC/'raw').glob('http_*.parquet'))
assert http_files, 'No HTTP partitions found.'
for p in http_files:
    h=prep_time(pd.read_parquet(p))
    h['domain']=(h['url'].astype(str).str.extract(r'(?i)^(?:https?://)?([^/]+)',expand=False).fillna('unknown').str.lower())
    q=(h.groupby(['user','day'],observed=True)
       .agg(http_count=('day','size'),http_afterhours=('afterhours','sum')).reset_index())
    count_parts.append(q)
    domain_parts.append(h[['user','day','domain']].drop_duplicates())
    print(p.name, f'{len(h):,} events')
    del h,q; gc.collect()
http_counts=(pd.concat(count_parts,ignore_index=True)
             .groupby(['user','day'],observed=True,as_index=False)[['http_count','http_afterhours']].sum())
dom=(pd.concat(domain_parts,ignore_index=True).drop_duplicates()
     .groupby(['user','day'],observed=True)['domain'].nunique().rename('http_distinct_domains').reset_index())
http_feats=http_counts.merge(dom,on=['user','day'],how='left')
del count_parts,domain_parts,http_counts,dom; gc.collect()
print('http days:',len(http_feats))


http_2026-01.parquet 1,152,377 events
http_2026-02.parquet 1,043,295 events
http_2026-03.parquet 1,150,359 events
http_2026-04.parquet 1,145,743 events
http_2026-05.parquet 1,152,757 events
http_2026-06.parquet 1,126,557 events
http days: 264890


## 3. Assemble the common 41-feature block

Create the active user-day table and join the 29 same-day technical features. Add twelve self-history deviations using shifted, preceding observations rather than the current or future day as a reference.

The feature names follow the Environment A contract. Matching names do not imply identical source distributions. The retained zero-valued `file_from_removable` column is one declared generator-specific difference.


In [7]:

# ============================================================
# 3. Active user-day spine + canonical M0-B block
# ============================================================
spine=(pd.concat([x[['user','day']] for x in [logon_feats,device_feats,file_feats,email_feats,http_feats]],ignore_index=True)
       .drop_duplicates().sort_values(['user','day']).reset_index(drop=True))
spine['dow']=spine['day'].dt.dayofweek.astype('int8')
spine['is_weekend']=(spine['dow']>=5).astype('int8')

feats=(spine.merge(logon_feats,on=['user','day'],how='left')
       .merge(device_feats,on=['user','day'],how='left')
       .merge(file_feats,on=['user','day'],how='left')
       .merge(email_feats,on=['user','day'],how='left')
       .merge(http_feats,on=['user','day'],how='left'))

hour_cols=['first_logon_hour','last_logoff_hour']
for c in SAME_DAY_FEATURES:
    if c not in feats.columns: feats[c]=0
count_cols=[c for c in SAME_DAY_FEATURES if c not in hour_cols]
feats[count_cols]=feats[count_cols].fillna(0)
feats[hour_cols]=feats[hour_cols].fillna(-1)
feats=feats[['user','day']+SAME_DAY_FEATURES].copy().sort_values(['user','day']).reset_index(drop=True)

# Causal self-history: preceding 30 active observations, at least 7 prior observations.
for c in BASE_FEATS:
    shifted=feats.groupby('user',observed=True)[c].shift(1)
    mean30=(shifted.groupby(feats['user'],observed=True).rolling(30,min_periods=7).mean().reset_index(level=0,drop=True))
    std30=(shifted.groupby(feats['user'],observed=True).rolling(30,min_periods=7).std().reset_index(level=0,drop=True))
    feats[f'{c}_z']=((feats[c]-mean30)/std30.replace(0,np.nan)).replace([np.inf,-np.inf],np.nan).fillna(0)

for c in R62_M0_FEATURES:
    feats[c]=pd.to_numeric(feats[c],errors='coerce').fillna(0).astype('float32')

# Chronological partition is determined from calendar only, not labels.
feats['day_number']=(feats['day']-START_DATE).dt.days+1
feats['partition']=np.select([feats.day_number<=TRAIN_END_DAY,feats.day_number<=VAL_END_DAY],['train','validation'],default='test')

assert list(feats[R62_M0_FEATURES].columns)==R62_M0_FEATURES
assert not feats.duplicated(['user','day']).any()
print('M0-B active user-days:',f'{len(feats):,}')
print(feats.groupby('partition').size())
print('file_from_removable nonzero rows:',int((feats.file_from_removable>0).sum()))


M0-B active user-days: 264,893
partition
test           87362
train         132574
validation     44957
dtype: int64
file_from_removable nonzero rows: 0


## 4. Attach the monthly organisational snapshots

Join on user and calendar month. In this generator, a snapshot applies from the first day of its month, so the exact-month attachment follows that generated effective-date rule.

Later snapshots are not moved backwards. This describes the synthetic record design; it is not an independent validation of organisational history in a real workplace.


In [8]:

# ============================================================
# 4. Organisational user-day attachment
# ============================================================
ldap_parts=[]
for p in sorted((SRC/'raw'/'LDAP').glob('*.parquet')):
    d=pd.read_parquet(p)
    d['snapshot']=p.stem
    d['month']=p.stem
    d['snapshot_date']=pd.to_datetime(d.get('snapshot_date',pd.Period(p.stem,freq='M').start_time))
    ldap_parts.append(d)
ldap=pd.concat(ldap_parts,ignore_index=True)
ldap['user']=ldap['user'].astype(str)

ctx=feats[['user','day','partition']].copy()
ctx['month']=ctx['day'].dt.to_period('M').astype(str)
attrs=['role','functional_unit','department','team','supervisor']
ctx=ctx.merge(ldap[['user','month','snapshot_date']+attrs],on=['user','month'],how='left')
for c in attrs: ctx[c]=ctx[c].fillna('UNKNOWN').astype(str)
assert len(ctx)==len(feats)
assert not ctx.duplicated(['user','day']).any()
ctx['snapshot_date']=pd.to_datetime(ctx['snapshot_date'])
future=(ctx['snapshot_date']>ctx['day']).fillna(False)
assert not future.any(), 'Future LDAP snapshot attached to earlier activity.'
ctx.to_parquet(QA/'organisational_attachment_provenance.parquet',index=False)
print('unknown roles:',int((ctx.role=='UNKNOWN').sum()))
print('roles:',sorted(ctx.role.unique().tolist()))


unknown roles: 0
roles: ['Cashier', 'Customer Service', 'IT Administrator', 'Payroll Finance', 'Store Manager', 'Warehouse Clerk']


## 5. Construct the curated context for M2-B

Build the 18-feature block: access proxy, role and department frequencies, hours deviations and peer-relative deviations. Frequencies and behavioural references are fitted on training rows and applied to the later partitions without updating those statistics.

The access value is a role-based proxy. Hours and peer features combine organisational grouping with technical behaviour; they are not independent permission or roster records.


In [9]:

# ============================================================
# 5a. Curated context functions
# ============================================================
ACCESS_RULES={
    'Cashier':1,
    'Warehouse Clerk':1,
    'Customer Service':1,
    'Payroll Finance':2,
    'Store Manager':2,
    'IT Administrator':3,
    'UNKNOWN':1,
}

def access_level(s):
    return s.map(ACCESS_RULES).fillna(1).astype('int32')

def fit_freq(df,col):
    t=df[['user',col]].drop_duplicates()
    denom=max(t.user.nunique(),1)
    return (t.groupby(col).user.nunique()/denom).to_dict()

def fit_expected_hours(tr,min_users=15):
    ok=tr[tr.first_logon_hour>=0].copy()
    gs=float(np.percentile(ok.first_logon_hour,10))
    end=ok.loc[ok.last_logoff_hour>=0,'last_logoff_hour']
    ge=float(np.percentile(end,90)) if len(end) else gs
    users=ok.groupby('role').user.nunique()
    prof=ok.groupby('role').agg(
        exp_start=('first_logon_hour',lambda s:np.percentile(s,10)),
        exp_end=('last_logoff_hour',lambda s:np.percentile(s[s>=0],90) if (s>=0).any() else np.nan))
    small=users[users<min_users].index
    prof.loc[prof.index.isin(small),['exp_start','exp_end']]=[gs,ge]
    return prof.fillna({'exp_start':gs,'exp_end':ge}).reset_index(),gs,ge

def fit_peer_stats(tr,min_users=15):
    x=tr.copy(); x['grp']=x.role.astype(str)+'|'+x.department.astype(str)
    n=x.groupby('grp').user.nunique(); small=set(n[n<min_users].index.astype(str))
    m=x.grp.isin(small); x.loc[m,'grp']=x.loc[m,'role'].astype(str)
    st=x.groupby('grp')[BASE_FEATS].agg(['mean','std'])
    st.columns=[f'{a}__{b}' for a,b in st.columns]
    glob={f'{c}__mean':float(x[c].mean()) for c in BASE_FEATS}
    glob.update({f'{c}__std':float(x[c].std()) for c in BASE_FEATS})
    return st.reset_index(),glob,small

def derive_context(ctx_attrs,tech,tag):
    use=['user','day','first_logon_hour','last_logoff_hour']+BASE_FEATS
    df=ctx_attrs.merge(tech[use],on=['user','day'],how='left')
    tr=df[df.partition=='train'].copy()
    out=df[['user','day']].copy()
    out['access_level']=access_level(df.role)
    rf=fit_freq(tr,'role'); dfreq=fit_freq(tr,'department')
    out['role_freq']=df.role.map(rf).fillna(min(rf.values())).astype('float32')
    out['dept_freq']=df.department.map(dfreq).fillna(min(dfreq.values())).astype('float32')
    hp,gs,ge=fit_expected_hours(tr)
    df=df.merge(hp,on='role',how='left')
    df['exp_start']=df.exp_start.fillna(gs); df['exp_end']=df.exp_end.fillna(ge)
    out['early_start_dev']=np.where(df.first_logon_hour>=0,(df.exp_start-df.first_logon_hour).clip(lower=0),0).astype('float32')
    out['late_end_dev']=np.where(df.last_logoff_hour>=0,(df.last_logoff_hour-df.exp_end).clip(lower=0),0).astype('float32')
    out['outside_expected_hours']=((out.early_start_dev>0)|(out.late_end_dev>0)).astype('int8')
    ps,glob,small=fit_peer_stats(tr)
    df['grp']=df.role.astype(str)+'|'+df.department.astype(str)
    m=df.grp.isin(small); df.loc[m,'grp']=df.loc[m,'role'].astype(str)
    df=df.merge(ps,on='grp',how='left')
    for c in BASE_FEATS:
        mu=df[f'{c}__mean'].fillna(glob[f'{c}__mean'])
        sd=df[f'{c}__std'].fillna(glob[f'{c}__std']).replace(0,np.nan)
        out[f'{c}_peer_z']=((df[c]-mu)/sd).replace([np.inf,-np.inf],np.nan).fillna(0).astype('float32')
    cols=[c for c in out if c not in ['user','day']]
    assert len(cols)==18,(tag,len(cols))
    return out,{'hours':hp,'peer_stats':ps,'role_freq':rf,'dept_freq':dfreq,'small_groups':sorted(small)}

ctx_cur,profiles=derive_context(ctx,feats,'M2')
ctx_cur.to_parquet(OUT/'context_curated_retail.parquet',index=False)
profiles['hours'].to_parquet(OUT/'expected_hours_profiles_retail.parquet',index=False)
profiles['peer_stats'].to_parquet(OUT/'peer_group_profiles_retail.parquet',index=False)
pd.DataFrame([{'role':k,'access_level':v} for k,v in ACCESS_RULES.items()]).to_csv(OUT/'access_level_rules_retail.csv',index=False)
print('M2 curated:',ctx_cur.shape)


M2 curated: (264893, 20)


## 6. Construct the raw comparator for M1-B

Use training-category indicators for role, functional unit and department, with frequency encodings for team and supervisor. These columns are added to the same technical block as the other configurations.

The raw block's size follows the simulated category inventory and need not match Environment A's M1 dimension. Column count is not the same as independent information content.


In [10]:

# ============================================================
# 6. Raw categorical comparator — matches the original design
# ============================================================
raw=ctx[['user','day']].copy()
TRAIN=ctx.partition.eq('train')
for col in ('role','functional_unit','department'):
    train_cats=ctx.loc[TRAIN,col].unique()
    oh=pd.get_dummies(pd.Categorical(ctx[col],categories=train_cats),dtype='int8')
    oh.columns=[f'{col}__{str(c)[:80]}' for c in oh.columns]
    raw=pd.concat([raw.reset_index(drop=True),oh.reset_index(drop=True)],axis=1)
for col in ('team','supervisor'):
    fr=fit_freq(ctx.loc[TRAIN],col)
    raw[f'{col}_freq']=ctx[col].map(fr).fillna(0).astype('float32')
assert not raw.columns.duplicated().any()
raw.to_parquet(OUT/'context_rawcat_retail.parquet',index=False)
print('M1 raw context feature columns:',raw.shape[1]-2)


M1 raw context feature columns: 20


## 7. Construct the monthly shuffled control for M3-B

Permute role–department pairs among distinct users within each calendar month, then rebuild the curated features using the shuffled assignments and training-fitted profiles.

The monthly category distribution is retained, but each employee's original sequence of assignments is not. Technical behaviour remains available to the hours and peer calculations. The control tests the implemented disruption of organisational correspondence, not removal of every behavioural dependency.


In [11]:

# ============================================================
# 7. Month-wise role+department shuffle
# ============================================================
rng=np.random.default_rng(M3_SEED)
base=ctx[['user','day','partition','month','role','department']].copy()
shuf_maps=[]
for month,g in base.groupby('month',sort=True):
    u=g[['user','role','department']].drop_duplicates('user').reset_index(drop=True)
    perm=rng.permutation(len(u))
    z=u[['user']].copy()
    z['role']=u.loc[perm,'role'].to_numpy()
    z['department']=u.loc[perm,'department'].to_numpy()
    z['month']=month
    shuf_maps.append(z)
sm=pd.concat(shuf_maps,ignore_index=True)
ctx_s=base[['user','day','partition','month']].merge(sm,on=['user','month'],how='left')
ctx_shuf,shuf_profiles=derive_context(ctx_s,feats,'M3')
ctx_shuf.to_parquet(OUT/'context_shuffled_retail.parquet',index=False)
print('M3 shuffled:',ctx_shuf.shape)


M3 shuffled: (264893, 20)


## 8. Save the feature tables, then attach separate labels

The four feature representations have now been constructed without reading malicious labels or scenario names. Save the label-free technical block before loading the answer records.

Outcome labels are written separately for evaluation. Scenario identities are retained for diagnostic reporting rather than added to the model feature matrices.


In [12]:

# ============================================================
# 8. Save label-free technical block, then load ground truth
# ============================================================
tech_out=feats[['user','day','partition']+R62_M0_FEATURES].copy()
tech_out.to_parquet(OUT/'features_cyberlog_retail.parquet',index=False)

# Ground truth is deliberately loaded only now.
ans_pq=SRC/'answers'/'malicious_user_days.parquet'
ans_csv=SRC/'answers'/'malicious_user_days.csv'
if ans_pq.exists():
    gt=pd.read_parquet(ans_pq)
elif ans_csv.exists():
    gt=pd.read_csv(ans_csv)
else:
    raise FileNotFoundError('malicious_user_days answer key not found')
gt['user']=gt.user.astype(str); gt['day']=pd.to_datetime(gt.day).dt.normalize()
labels=tech_out[['user','day','partition']].copy()
labels=labels.merge(gt[['user','day']].drop_duplicates().assign(label=np.int8(1)),on=['user','day'],how='left')
labels['label']=labels.label.fillna(0).astype('int8')
labels.to_parquet(OUT/'labels_retail.parquet',index=False)

# Scenario names remain in a separate diagnostics-only file.
scen=gt[['user','day','partition','scenario']].drop_duplicates()
scen.to_parquet(QA/'scenario_labels_PRIVATE.parquet',index=False)

print(labels.groupby('partition').agg(user_days=('day','size'),malicious_days=('label','sum'),users=('user','nunique')))
print('malicious accounts by partition:')
print(labels[labels.label==1].groupby('partition').user.nunique())


            user_days  malicious_days  users
partition                                   
test            87362             180   2000
train          132574               0   2000
validation      44957              90   2000
malicious accounts by partition:
partition
test          60
validation    30
Name: user, dtype: int64


## 9. Check the feature contract and package the outputs

Review the structural checks, feature names, dimensions, partitions and label separation. Save the fitted profiles, feature contract, run summary and output hashes, then create the 7R-B ZIP package for 7R-C.

These checks verify the declared input/output contract. They do not assess detector performance or decide H1-B.


In [13]:

# ============================================================
# 9. QA + contracts
# ============================================================
checks=[]
def check(name,cond,detail=''):
    cond=bool(cond); checks.append({'check':name,'passed':cond,'detail':str(detail)}); print(('PASS' if cond else 'FAIL')+':',name,detail); return cond

check('7R-A source hashes all passed',ver.passed.all(),f'{int(ver.passed.sum())}/{len(ver)}')
check('M0 contains exactly 41 canonical technical features',list(tech_out[R62_M0_FEATURES].columns)==R62_M0_FEATURES)
check('M0 numeric block has no NaN',int(tech_out[R62_M0_FEATURES].isna().sum().sum())==0)
check('M0 numeric block is finite',np.isfinite(tech_out[R62_M0_FEATURES].to_numpy(dtype='float32')).all())
check('M2 has exactly 18 context features',ctx_cur.shape[1]-2==18,ctx_cur.shape)
check('M3 has exactly 18 context features',ctx_shuf.shape[1]-2==18,ctx_shuf.shape)
check('M1/M2/M3 row counts match M0',len(raw)==len(ctx_cur)==len(ctx_shuf)==len(tech_out))
check('feature keys are unique',not tech_out.duplicated(['user','day']).any())
check('no future LDAP snapshot attached',not (ctx.snapshot_date>ctx.day).fillna(False).any())
check('training has zero malicious user-days',int(labels.loc[labels.partition=='train','label'].sum())==0)
check('validation malicious days = 90',int(labels.loc[labels.partition=='validation','label'].sum())==90)
check('test malicious days = 180',int(labels.loc[labels.partition=='test','label'].sum())==180)
check('validation malicious accounts = 30',labels[(labels.partition=='validation')&(labels.label==1)].user.nunique()==30)
check('test malicious accounts = 60',labels[(labels.partition=='test')&(labels.label==1)].user.nunique()==60)
for name,df in [('M0',tech_out),('M1',raw),('M2',ctx_cur),('M3',ctx_shuf)]:
    bad=[c for c in df.columns if any(x in c.lower() for x in ['label','scenario','malicious','insider'])]
    check(f'{name} contains no target/scenario feature fields',len(bad)==0,bad)
check('file_from_removable structural-zero fact recorded',int((tech_out.file_from_removable!=0).sum())==0)

qa=pd.DataFrame(checks); qa.to_csv(QA/'7rb_structural_checks.csv',index=False)
assert qa.passed.all(),'7R-B structural QA failed.'

contract={
 'stage':'7R-B','source_generator_manifest':manifest,
 'm0_feature_count':41,'m0_feature_names':R62_M0_FEATURES,
 'same_day_feature_count':29,'self_history_feature_count':12,
 'm1_context_feature_count':int(raw.shape[1]-2),
 'm2_context_feature_count':18,'m2_context_feature_names':[c for c in ctx_cur if c not in ['user','day']],
 'm3_context_feature_count':18,
 'access_proxy_rules':ACCESS_RULES,
 'm3_shuffle':'role+department jointly shuffled within calendar month; seed 42',
 'declared_source_detail':{'file_from_removable':'structurally zero in generator v1.0'},
 'answer_key_loaded_after_feature_construction':True,
 'private_latent_plan_used':False,
 'model_trained':False,'h1_b_verdict':None
}
json.dump(contract,open(OUT/'retail_7rb_feature_contract.json','w'),indent=2)

summary={
 'source_condition':manifest.get('condition'),'source_seed':manifest.get('seed'),'source_run_mode':manifest.get('run_mode'),
 'active_user_days':int(len(tech_out)),'employees':int(tech_out.user.nunique()),
 'partition_user_days':{p:int((labels.partition==p).sum()) for p in ['train','validation','test']},
 'malicious_user_days':{p:int(labels.loc[labels.partition==p,'label'].sum()) for p in ['train','validation','test']},
 'malicious_accounts':{p:int(labels.loc[(labels.partition==p)&(labels.label==1),'user'].nunique()) for p in ['train','validation','test']},
 'm0_features':41,'m1_context_features':int(raw.shape[1]-2),'m2_context_features':18,'m3_context_features':18,
 'model_trained':False,'h1_b_verdict':None
}
json.dump(summary,open(OUT/'retail_7rb_summary.json','w'),indent=2)

# Hash all outputs except the manifest being written.
rows=[]
for p in sorted(OUT.rglob('*')):
    if p.is_file() and p.name!='retail_7rb_output_sha256.csv':
        rows.append({'relative_path':str(p.relative_to(OUT)),'size_bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(rows).to_csv(OUT/'retail_7rb_output_sha256.csv',index=False)

print('\n7R-B COMPLETE')
print(json.dumps(summary,indent=2))
print('\nNO MODEL WAS TRAINED. NO H1-B VERDICT WAS MADE.')


PASS: 7R-A source hashes all passed 25/25
PASS: M0 contains exactly 41 canonical technical features 
PASS: M0 numeric block has no NaN 
PASS: M0 numeric block is finite 
PASS: M2 has exactly 18 context features (264893, 20)
PASS: M3 has exactly 18 context features (264893, 20)
PASS: M1/M2/M3 row counts match M0 
PASS: feature keys are unique 
PASS: no future LDAP snapshot attached 
PASS: training has zero malicious user-days 
PASS: validation malicious days = 90 
PASS: test malicious days = 180 
PASS: validation malicious accounts = 30 
PASS: test malicious accounts = 60 
PASS: M0 contains no target/scenario feature fields []
PASS: M1 contains no target/scenario feature fields []
PASS: M2 contains no target/scenario feature fields []
PASS: M3 contains no target/scenario feature fields []
PASS: file_from_removable structural-zero fact recorded 

7R-B COMPLETE
{
  "source_condition": "B1_structured",
  "source_seed": 2105,
  "source_run_mode": "FINAL",
  "active_user_days": 264893,
  "em

In [14]:

# ============================================================
# 10. Package 7R-B outputs
# ============================================================
zip_path=shutil.make_archive(str(WORK/'retail_7rb_results'),'zip',root_dir=OUT)
print('Created:',zip_path)
print('Upload retail_7rb_results.zip for the 7R-C modelling/evaluation stage.')


Created: /kaggle/working/retail_7rb_results.zip
Upload retail_7rb_results.zip for the 7R-C modelling/evaluation stage.
